In [16]:
# Installs for RunPod
!pip install transformers datasets peft jaxtyping accelerate matplotlib scikit-learn numpy > /dev/null

In [2]:
# Load Hugging Face access token
# Manage your secrets from the "Add-ons" menu in the top navigation of the editor
import os
os.environ["HF_TOKEN"] = "REDACTED"

In [3]:
'''
# Load model and tokenizer
'''
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    padding_side="right",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()
print(f"Model loaded. Layers: {model.config.num_hidden_layers}")  # 32 for Llama-3.1-8B

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Model loaded. Layers: 28


In [4]:
import torch.nn as nn
from typing import Dict, List, Optional, Union, Tuple, Callable
from peft import PeftModel, get_peft_model
from pathlib import Path
from jaxtyping import Float, Int
from torch import Tensor
from transformers import PreTrainedModel
import contextlib
import functools


def get_model_hidden_size(model: PreTrainedModel) -> int:
    """
    Get the hidden size of a transformer model.
    
    Args:
        model: The transformer model
        
    Returns:
        Hidden size of the model
    """
    # Handle PeftModel
    if isinstance(model, PeftModel):
        base_model = model.get_base_model()
    else:
        base_model = model
    
    # Infer from the model structure
    if hasattr(base_model, 'model') and hasattr(base_model.model, 'embed_tokens'):
        return base_model.model.embed_tokens.weight.shape[1]
    
    raise ValueError(f"Could not determine hidden size for model type {type(base_model)}")


def get_model_layers(model: PreTrainedModel) -> List[nn.Module]:
    """
    Get the list of transformer layers from a model.
    
    Args:
        model: The transformer model
    
    Returns:
        List of layer modules
    """
    # Handle PeftModel by getting the base model
    if isinstance(model, PeftModel):
        base_model = model.get_base_model()
    else:
        base_model = model
    
    # Common patterns for accessing layers in different model architectures
    if hasattr(base_model, 'model') and hasattr(base_model.model, 'layers'):
        # LLaMA, Mistral, etc.
        return list(base_model.model.layers)
    elif hasattr(base_model, 'transformer') and hasattr(base_model.transformer, 'h'):
        # GPT-2, GPT-J, etc.
        return list(base_model.transformer.h)
    elif hasattr(base_model, 'encoder') and hasattr(base_model.encoder, 'layer'):
        # BERT, RoBERTa, etc.
        return list(base_model.encoder.layer)
    elif hasattr(base_model, 'gpt_neox') and hasattr(base_model.gpt_neox, 'layers'):
        # GPT-NeoX
        return list(base_model.gpt_neox.layers)
    else:
        raise ValueError(f"Unknown model architecture: {type(base_model)}")


@contextlib.contextmanager
def add_hooks(
    module_forward_pre_hooks: List[Tuple[nn.Module, Callable]],
    module_forward_hooks: List[Tuple[nn.Module, Callable]],
    **kwargs
):
    """
    Context manager for temporarily adding forward hooks to a model.
    
    Args:
        module_forward_pre_hooks: List of (module, hook_fn) pairs for pre-hooks
        module_forward_hooks: List of (module, hook_fn) pairs for forward hooks
        **kwargs: Additional arguments passed to hook functions
    """
    handles = []
    try:
        # Register pre-hooks
        for module, hook in module_forward_pre_hooks:
            partial_hook = functools.partial(hook, **kwargs)
            handles.append(module.register_forward_pre_hook(partial_hook))
        
        # Register forward hooks
        for module, hook in module_forward_hooks:
            partial_hook = functools.partial(hook, **kwargs)
            handles.append(module.register_forward_hook(partial_hook))
        
        yield
    finally:
        # Remove all hooks
        for handle in handles:
            handle.remove()


class ValueHeadProbe(nn.Module):
    """
    A probe that hooks into a specific layer of a language model and uses a linear
    head to predict per-token binary classification (e.g., hallucination detection).
    
    This probe:
    1. Attaches a forward hook to capture hidden states from a specified layer
    2. Applies a linear transformation to predict hallucination scores
    3. Can be used with or without LoRA adapters
    """
    
    def __init__(
        self,
        model: Union[AutoModelForCausalLM, PeftModel],
        layer_idx: Optional[int] = None,
        path: Optional[Union[str, Path]] = None,
    ):
        """
        Initialize the ValueHeadProbe.
        
        Args:
            model: The language model (with or without LoRA adapters)
            layer_idx: Optional index of the layer to attach the probe to (if not specified we'll infer it from the previously saved probe configuration file)
            path: Optional path to load pre-trained probe weights from
        """
        super().__init__()

        assert layer_idx or path, "Either path or layer index must be provided, otherwise we can't infer the layer where we should hook the value head to."
        
        if path:
            saved_config = json.load(open(path / "probe_config.json"))

            if layer_idx is None:
                layer_idx = saved_config["layer_idx"]
            else:
                # We check that if a layer_idx is provided, it matches the one given in the previously saved config
                assert layer_idx == saved_config["layer_idx"]

        hidden_size = get_model_hidden_size(model)
        model_layers = get_model_layers(model)

        self.model = model
        self.layer_idx = layer_idx
        self.target_module = model_layers[layer_idx]
        self.target_layer_name = self.target_module.__class__.__name__
        
        if not isinstance(model, PeftModel):
            print("WARNING: Model is not a PeftModel. Remember to add LoRA adapters if needed.")
        
        # Initialize the value head (linear layer)
        if path:
            # Load pre-trained weights if path is provided
            self.value_head, _ = ValueHeadProbe.load_head(
                path,
                device=model.device,
                dtype=model.dtype
            )
        else:
            self.value_head = nn.Linear(
                hidden_size, 1, 
                device=model.device, 
                dtype=model.dtype
            )
            print(f"WARNING: Using seed=42 for the initialization of the probe")
            torch.manual_seed(42)
            self._initialize_weights()
        
        # Initialize hook state
        self._hooked_hidden_states: Optional[torch.Tensor] = None
        self._hook_fn = self._get_hook_fn()
    
    def _initialize_weights(self):
        """Initialize the value head weights with small random values."""
        with torch.no_grad():
            self.value_head.weight.data.normal_(mean=0.0, std=0.01)
            if self.value_head.bias is not None:
                self.value_head.bias.data.zero_()
    
    def _get_hook_fn(self):
        """
        Forward hook to capture hidden states from target layer.
        `module_output` is typically [batch_size, seq_len, hidden_dim].
        We do NOT detach it, so gradients can flow back.
        """
        def hook_fn(module, module_input, module_output):
            if isinstance(module_input, tuple) and module_output[0].ndim == 3:
                self._hooked_hidden_states = module_output[0]
            else:
                self._hooked_hidden_states = module_output
        return hook_fn
    
    def forward(
        self,
        input_ids: Int[Tensor, 'batch_size seq_len'],
        attention_mask: Optional[Int[Tensor, 'batch_size seq_len']] = None,
        labels: Optional[Int[Tensor, 'batch_size seq_len']] = None,
        **kwargs
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass through the model with probe attached.
        
        Args:
            input_ids: Input token IDs [batch_size, seq_len]
            attention_mask: Attention mask [batch_size, seq_len]
            labels: Optional labels for language modeling loss
            **kwargs: Additional arguments passed to the model
        
        Returns:
            Dictionary containing:
                - lm_logits: Language model output logits [batch_size, seq_len, vocab_size]
                - probe_logits: Probe output logits [batch_size, seq_len, 1]
                - lm_loss: Language modeling loss (if labels provided)
        """
        # Reset hooked hidden states
        self._hooked_hidden_states = None
        
        # Set up hooks
        fwd_hooks = [(self.target_module, self._hook_fn)]
        
        with add_hooks(module_forward_pre_hooks=[], module_forward_hooks=fwd_hooks):
            # Forward pass through the model
            outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
                output_hidden_states=False,
                **kwargs
            )
        
        # Check that hidden states were captured
        if self._hooked_hidden_states is None:
            raise RuntimeError("Failed to capture hidden states from target layer")

        probe_logits: Float[Tensor, 'batch_size seq_len 1'] = self.value_head(
            self._hooked_hidden_states.to(self.value_head.weight.device)
        )

        return {
            "lm_logits": outputs.logits,
            "probe_logits": probe_logits,
            "lm_loss": outputs.loss if hasattr(outputs, 'loss') else None
        }
    
    def save(self, path: Union[str, Path]):
        """
        Save the probe to disk.
        
        Args:
            path: Directory to save the probe to
        """
        os.makedirs(path, exist_ok=True)
        
        # Save LoRA adapters if present
        if isinstance(self.model, PeftModel):
            self.model.save_pretrained(path)
        
        # Save value head weights
        torch.save(
            self.value_head.state_dict(),
            path / "probe_head.bin"
        )
        
        # Save configuration
        probe_config = {
            "target_layer_name": self.target_module.__class__.__name__,
            "layer_idx": self.layer_idx,
            "hidden_size": self.value_head.in_features
        }
        with open(path / "probe_config.json", 'w') as f:
            json.dump(probe_config, f, indent=4)
        
        print(f"Probe saved to {path}")
    
    @property
    def device(self):
        """Get the device of the model."""
        return self.model.device
    
    @classmethod
    def load_head(
        cls,
        path: Path,
        device: str = 'cuda',
        dtype: torch.dtype = torch.bfloat16
    ) -> Tuple[nn.Module, int]:
        """
        Loads the linear value head from the given path.
        
        Args:
            path: Path to the probe directory
            device: Device to load the probe head on
            dtype: Data type for the probe head
            
        Returns:
            Tuple of (probe_head, layer_idx)
        """
        with open(path / "probe_config.json") as f:
            probe_config = json.load(f)

        hidden_size = probe_config['hidden_size']
        probe_layer_idx = probe_config['layer_idx']

        probe_head = nn.Linear(hidden_size, 1, device=device, dtype=dtype)
        
        state_dict = torch.load(
            path / "probe_head.bin",
            map_location=device,
            weights_only=True
        )
        probe_head.load_state_dict(state_dict)
        
        return probe_head, probe_layer_idx

In [5]:
'''
Setup the probe.
'''
PROBE_LAYER = 12

# Freeze all parameters of the base model
for _, param in model.named_parameters():
    param.requires_grad = False

probe = ValueHeadProbe(model, layer_idx=PROBE_LAYER)

print("Parameters that will be trained:")
trainable_params = 0
total_params = 0
for name, param in model.named_parameters():
    if param.requires_grad:
        trainable_params += param.numel()
        print(f"  - {name}: shape {param.shape}, device {param.device}")
    total_params += param.numel()    
trainable_params_percentage = 100 * trainable_params / total_params
print(f"\nTotal trainable parameters: {trainable_params:,} ({trainable_params_percentage:.2f}%)")
print(f"Total parameters: {total_params:,}")

Parameters that will be trained:

Total trainable parameters: 0 (0.00%)
Total parameters: 3,212,749,824


In [6]:
from copy import deepcopy
from dataclasses import dataclass
from torch.utils.data import Dataset
import random
from tqdm import tqdm


@dataclass
class AnnotatedSpan:
    """A text span with its hallucination label."""    
    span: str  # The span text
    label: float  # 1.0 for hallucination, 0.0 for supported, -100.0 for ignored
    index: int  # Start index in the completion


@dataclass
class ProbingItem:
    """A single item containing prompt, completion and annotated spans."""    
    prompt: str
    completion: str
    spans: List[AnnotatedSpan]


class TokenizedProbingDataset(Dataset):
    """Dataset for probing model activations with annotated spans."""
    
    def __init__(
        self,
        items: List[ProbingItem],
        config,
        tokenizer: AutoTokenizer,
    ):
        self.config = config
        self.tokenizer = tokenizer
        self.items = deepcopy(items)
        self.processed_items = [None] * len(items)
        self.debug_mode = False
        self.print_first_example = False
        
        self._num_skipped_spans: int = 0
        self._num_added_spans: int = 0
        
        if self.config.shuffle:
            self._shuffle_items()
        
        # Limit samples if specified (do this after shuffling)
        if self.config.max_num_samples:
            self.items = self.items[:self.config.max_num_samples]
            self.processed_items = self.processed_items[:self.config.max_num_samples]
        
        if not self.config.process_on_the_fly:
            self._process_items()
    
    def _process_items(self):
        """Pre-process all items in the dataset."""
        for i, item in tqdm(enumerate(self.items), desc=f"Processing items ({self.config.dataset_id})", total=len(self.items)):
            if i == 0 and self.print_first_example:
                self.debug_mode = True
            else:
                self.debug_mode = False
            
            processed_item = self._process_item(item)
            if processed_item:
                self.processed_items[i] = processed_item
        
        print(f"Dataset {self.config.dataset_id} stats:")
        print(f"\t- Number of added spans: {self._num_added_spans}")
        print(f"\t- Number of skipped spans: {self._num_skipped_spans} / {self._num_added_spans + self._num_skipped_spans}")
        print(f"\t- Total number of items: {len(self.items)}")
    
    def _process_item(self, item: ProbingItem) -> Dict:
        """Process a single example into tokenized format with labels."""
        conversation = [
            {'role': 'user', 'content': item.prompt},
            {'role': 'assistant', 'content': item.completion}
        ]
        full_text = self.tokenizer.apply_chat_template(conversation, tokenize=False)
        
        if self.tokenizer.bos_token and self.tokenizer.bos_token in full_text:
            full_text = full_text.replace(self.tokenizer.bos_token, '')
        
        encoding = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.config.max_length,
            padding='max_length',
            return_tensors='pt',
            padding_side='right'
        )
        
        input_ids: Int[Tensor, "seq_len"] = encoding["input_ids"][0]
        attention_mask: Int[Tensor, "seq_len"] = encoding["attention_mask"][0]
        
        labels, weights, pos_spans, neg_spans = self._compute_positional_labels(
            input_ids=input_ids,
            item=item
        )
        
        input_str: str = self.tokenizer.decode(input_ids)
        assistant_tokens_slice = find_assistant_tokens_slice(input_ids, input_str, self.tokenizer)
        completion_start_idx = assistant_tokens_slice.stop
        
        lm_labels = input_ids.clone()
        lm_labels[:completion_start_idx] = -100  # ignore all tokens in the prompt
        lm_labels[attention_mask == 0] = -100  # ignore padding tokens

        return {
            "input_ids": input_ids,  # Int[Tensor, "seq_len"]
            "attention_mask": attention_mask,  # Int[Tensor, "seq_len"]
            "classification_labels": labels,  # Float[Tensor, "seq_len"]
            "classification_weights": weights,  # Float[Tensor, "seq_len"]
            "pos_spans": pos_spans,  # List[List[int]]
            "neg_spans": neg_spans,  # List[List[int]]
            "lm_labels": lm_labels,  # Int[Tensor, "seq_len"]
        }
    
    def print_token_labels(
        self,
        input_ids: torch.Tensor,
        positive_indices: List[int],
        negative_indices: List[int],
        ignore_indices: List[int],
        spans: List[AnnotatedSpan]
    ):
        """Debug method to print how tokens have been labeled."""

        tokens = [self.tokenizer.decode(tok) for tok in input_ids]

        print(f"================================================")
        print(f"Number of spans: {len(spans)}")
        print(f"Number of non-factual (hallucinated) spans: {len([f for f in spans if f.label == 1.0])}")
        print(f"Number of N/A spans: {len([f for f in spans if f.label == -100])}")
        print(f"Number of factual spans: {len([f for f in spans if f.label == 0.0])}")
        print(f"Legend: red - positive, green - negative, blue - ignored")

        for i, token in enumerate(tokens):
            if token == self.tokenizer.eos_token:
                continue

            if i in positive_indices:
                print(colored(token, 'red'), end='')
            elif i in negative_indices:
                print(colored(token, 'green'), end='')
            elif i in ignore_indices:
                print(colored(token, 'blue'), end='')
            else:
                print(token, end='')

        print(f"================================================")
    
    def _compute_positional_labels(
        self,
        input_ids: torch.Tensor,
        item: ProbingItem
    ) -> Tuple[torch.Tensor, torch.Tensor, List[List[int]], List[List[int]]]:
        """
        Computes positional labels for a sequence of tokens based on annotated spans.
        
        For each span in the example:
        - If it's a hallucination (label 1.0): Sets span tokens to 1.0 and nearby tokens within
          ignore_buffer to -100.0
        - If it's a supported fact (label 0.0): Sets span tokens to 0.0 and nearby tokens within
          ignore_buffer to -100.0
        - If it's unlabeled/undecided (label -100.0): Sets span tokens to -100.0
        
        Args:
            input_ids: Input token IDs
            item: ProbingItem containing the spans and their labels
            
        Returns:
            Tuple of (labels, weights, pos_spans, neg_spans)
        """
        input_str: str = self.tokenizer.decode(input_ids)
        completion: str = item.completion
        
        positive_indices: List[int] = []    # indices of hallucinated spans
        negative_indices: List[int] = []    # indices of supported spans
        ignore_indices: List[int] = []      # indices to ignore in training
        
        positive_spans: List[List[int]] = []
        negative_spans: List[List[int]] = []
        
        def get_nearby_indices(span_indices: List[int]) -> List[int]:
            left_window = list(range(max(0, span_indices[0] - self.config.ignore_buffer), span_indices[0]))
            right_window = list(range(span_indices[-1] + 1, min(len(input_ids), span_indices[-1] + 1 + self.config.ignore_buffer)))
            return left_window + right_window
        
        # Find assistant tokens slice to know where to start looking for spans
        assistant_tokens_slice = find_assistant_tokens_slice(
            input_ids,
            input_str,
            self.tokenizer
        )
        completion_start_idx = assistant_tokens_slice.stop
        cur_idx = assistant_tokens_slice.stop
        
        # Sort spans by their index in the text
        spans = sorted(item.spans, key=lambda x: x.index)
        
        for span in spans:
            if span.span not in input_str:
                self._num_skipped_spans += 1
                continue
            
            try:
                # First try to find the span after the assistant tokens
                positions_slice = find_string_in_tokens(span.span, input_ids[cur_idx:], self.tokenizer)
                positions_slice = slice(positions_slice.start + cur_idx, positions_slice.stop + cur_idx)
            except (AssertionError, ValueError):
                try:
                    # If not found, try the whole input_ids
                    print(f"Repeating position_slice search on all tokens after failing to find span {repr(span.span)} in input_ids[cur_idx:]: {repr(self.tokenizer.decode(input_ids[cur_idx:]))[:50]}...")
                    positions_slice = find_string_in_tokens(span.span, input_ids, self.tokenizer)
                except (AssertionError, ValueError) as e:
                    print(f"Span {repr(span.span)} not found in input_ids, skipping entity")
                    self._num_skipped_spans += 1
                    continue
            
            if positions_slice is None:
                continue
            
            span_indices = slice_to_list(positions_slice, len(input_ids))
            if not span_indices:
                continue
            
            cur_idx = positions_slice.start
            
            if self.config.last_span_token:
                # If last_span_token is true, only use the last token of the span
                span_indices = [span_indices[-1]]
            
            # Get indices of tokens to ignore around this span
            nearby_indices = get_nearby_indices(span_indices)
            
            if span.label == 1.0:  # Hallucination
                positive_indices.extend(span_indices)
                ignore_indices.extend(nearby_indices)
                positive_spans.append([span_indices[0], span_indices[-1]])
            elif span.label == 0.0:  # Supported
                negative_indices.extend(span_indices)
                negative_spans.append([span_indices[0], span_indices[-1]])
            else:  # -100.0 (ignored)
                ignore_indices.extend(span_indices)
            
            self._num_added_spans += 1
        
        # Remove duplicates and sort
        positive_indices = sorted(list(set(positive_indices)))
        negative_indices = sorted(list(set(negative_indices) - set(positive_indices)))
        ignore_indices = sorted(list(set(ignore_indices) - set(positive_indices) - set(negative_indices)))
        
        # Initialize labels and weights
        default_label = -100.0 if self.config.default_ignore else 0.0
        labels = torch.full((len(input_ids),), default_label, dtype=torch.float32)
        
        # Set labels and weights
        labels[input_ids == self.tokenizer.pad_token_id] = -100.0
        labels[:completion_start_idx] = -100.0
        labels[ignore_indices] = -100.0
        labels[positive_indices] = 1.0
        labels[negative_indices] = 0.0
        
        weights = torch.full((len(input_ids),), 1.0, dtype=torch.float32)
        weights[ignore_indices] = 0.0  # N/A weight
        weights[positive_indices] = self.config.pos_weight
        weights[negative_indices] = self.config.neg_weight
        
        if self.debug_mode:
            self.print_token_labels(input_ids, positive_indices, negative_indices, ignore_indices, spans)
        
        return labels, weights, positive_spans, negative_spans
    
    def _shuffle_items(self):
        """Shuffle the items using the configured seed."""
        random.seed(self.config.seed)
        random.shuffle(self.items)
        random.seed(self.config.seed)
        random.shuffle(self.processed_items)
    
    def __len__(self):
        return len(self.items)
    
    def __getitem__(self, idx):
        if self.config.process_on_the_fly and self.processed_items[idx] is None:
            self.processed_items[idx] = self._process_item(self.items[idx])
        
        return self.processed_items[idx]
    
    def __add__(self, other):
        """
        Concatenate two TokenizedProbingDataset instances.
        
        Args:
            other: Another TokenizedProbingDataset instance to concatenate
            
        Returns:
            TokenizedProbingDataset: A new dataset containing items from both
        """
        if not isinstance(other, TokenizedProbingDataset):
            raise TypeError(f"Can only concatenate with another TokenizedProbingDataset, got {type(other)}")
        
        if self.config.max_length != other.config.max_length:
            raise ValueError("Can't concatenate datasets of different token lengths")
        
        if self.config.shuffle != other.config.shuffle:
            raise ValueError("Can't concatenate datasets if one of them (but not the other) are shuffled")
        
        # Create a new dataset with combined items
        combined_items = self.items + other.items
        combined_processed_items = self.processed_items + other.processed_items
        
        # Use the configuration from the first dataset
        new_dataset = TokenizedProbingDataset(
            items=[],  # we don't want to recompute everything again
            tokenizer=self.tokenizer,
            config=self.config,
        )
        
        new_dataset.items = combined_items
        new_dataset.processed_items = combined_processed_items
        
        if self.config.shuffle:
            new_dataset._shuffle_items()
        
        return new_dataset
    
    def __radd__(self, other):
        return self.__add__(other)


"""Dataset converters for different HuggingFace dataset formats to probing format."""

from typing import Callable, Dict, List, Optional
from datasets import Dataset


# Mapping from text labels to numeric values for probe training
_MAP_LABEL_TO_SCALAR = {
    'Not Supported': 1.0,
    'NS': 1.0,  # the probe should output 1.0 on text containing unsupported claims
    'Insufficient Information': 1.0,  # the probe should also output 1.0 if the label is 'Insufficient Information'
    'Supported': 0.0,
    'S': 0.0,
    'N/A': -100.0,
    None: -100.0
}

def prepare_longform_dataset(dataset: Dataset) -> List[ProbingItem]:
    """Prepare dataset from the one-shot pipeline labeling format."""
    probing_items: List[ProbingItem] = []

    for hf_item in dataset:
        prompt = hf_item['conversation'][-2]['content']
        completion = hf_item['conversation'][-1]['content']
        annotations: List[dict] = hf_item['annotations']

        annotated_spans: List[AnnotatedSpan] = []

        for entity in annotations:
            if entity is None or 'index' not in entity or not isinstance(entity['index'], int):
                continue

            entity_text = entity['span']
            label = entity['label']
            idx = entity['index']
            
            if idx is None:
                print(f"Entity {repr(entity_text)}'s idx set to None, discarding entity")
                continue
            elif not entity_text or entity_text not in completion:
                print(f"Entity {repr(entity_text)} not found in completion, discarding entity")
                continue

            annotated_spans.append(
                AnnotatedSpan(
                    span=entity_text,
                    label=_MAP_LABEL_TO_SCALAR[label],
                    index=idx
                )
            )

        probing_items.append(
            ProbingItem(
                prompt=prompt,
                completion=completion,
                spans=annotated_spans
            )
        )

    return probing_items


def prepare_longform_dataset_old_format(dataset: Dataset) -> List[ProbingItem]:
    """Prepare dataset from the one-shot pipeline labeling format."""
    probing_items: List[ProbingItem] = []

    for hf_item in dataset:
        prompt = hf_item['conversation'][0]['content']
        completion = hf_item['completion'] if 'completion' in hf_item else hf_item['conversation'][-1]['content']
        annotations: List[dict] = hf_item['verified_entities']

        annotated_spans: List[AnnotatedSpan] = []

        for entity in annotations:
            if entity is None or 'idx' not in entity or not isinstance(entity['idx'], int):
                continue

            entity_text = entity['text']
            label = entity['label']
            idx = entity['idx']
            
            if idx is None:
                print(f"Entity {repr(entity_text)}'s idx set to None, discarding entity")
                continue
            elif not entity_text or entity_text not in completion:
                print(f"Entity {repr(entity_text)} not found in completion, discarding entity")
                continue

            annotated_spans.append(
                AnnotatedSpan(
                    span=entity_text,
                    label=_MAP_LABEL_TO_SCALAR[label],
                    index=idx
                )
            )

        probing_items.append(
            ProbingItem(
                prompt=prompt,
                completion=completion,
                spans=annotated_spans
            )
        )

    return probing_items


def prepare_triviaqa(dataset: Dataset) -> List[ProbingItem]:
    """
    Pre-processes TriviaQA dataset.
    The greedy completion (labeled by an LLM) is at `gt_completion`
    The label is at `llm_judge_label` and it's a string containing `S`, `NS`, `N/A` or some undefined string
    The annotated spans will be the *whole completion*
    """
    assert 'question' in dataset[0] or 'conversation' in dataset[0]

    LABEL_FIELD: str = 'llm_judge_label' if 'llm_judge_label' in dataset.features else 'label'
    COMPLETION_FIELD: str = 'gt_completion'
    VALID_LABELS: List[str] = ['S', 'NS', 'N/A']
    EXACT_ANSWER_FIELD: str = 'exact_answer'

    probing_items = []
    for item in dataset:
        if item[LABEL_FIELD] not in VALID_LABELS:
            print(f"Invalid label {item[LABEL_FIELD]} for item, skipping")
            continue

        prompt = item['question'] if 'question' in item else item['conversation'][0]['content']
        completion = item[COMPLETION_FIELD]
        exact_answer = item[EXACT_ANSWER_FIELD] if EXACT_ANSWER_FIELD in item else ""

        if exact_answer is None or exact_answer not in completion:
            print(f"Exact answer {repr(exact_answer)} not found in completion {repr(completion)}")
            return None

        exact_answer_start_idx = completion.find(exact_answer)
        
        # The whole completion is labeled with the given label
        label_value: float = _MAP_LABEL_TO_SCALAR[item[LABEL_FIELD]]
        
        annotated_spans = [
            AnnotatedSpan(
                span=exact_answer,
                label=label_value,
                index=exact_answer_start_idx
            )
        ]

        probing_items.append(
            ProbingItem(
                prompt=prompt,
                completion=completion,
                spans=annotated_spans
            )
        )

    return probing_items


def prepare_synthetic(dataset: Dataset) -> List[ProbingItem]:
    """Loads the synthetic dataset from the hub."""
    FIELD = 'probing_item_with_hallucinations'

    probing_items = []
    for i, item in enumerate(dataset):
        probing_item = item[FIELD]
        annotated_spans = [
            AnnotatedSpan(
                span=span['text'],
                label=span['label'],
                index=span['start_idx']
            )
            for span in probing_item['spans']
        ]

        # Sort spans by their index in the text
        completion = probing_item['completion']

        if len(completion) <= 500:
            print(f"For item {i} completion is too short ({len(completion)} characters): {repr(completion)}")
            continue

        annotated_spans = sorted(annotated_spans, key=lambda x: x.index)

        if not all(completion[span.index:span.index+len(span.span)] == span.span for span in annotated_spans):
            print(f"For item {i} spans are not aligned with the completion")
            for span in annotated_spans:
                if completion[span.index:span.index+len(span.span)] != span.span:
                    print(f"- Span: {span.span} | {span.index} | {span.label}")
            continue

        probing_items.append(ProbingItem(
            prompt=probing_item['prompt'],
            completion=probing_item['completion'],
            spans=annotated_spans
        ))
    return probing_items


def get_prepare_function(
    hf_repo: str,
    subset: Optional[str] = None
) -> Callable[[Dataset], List[ProbingItem]]:
    """Get the appropriate preparation function based on dataset name."""
    if 'one_shot_pipeline' in str(subset) or 'hallucination-heads' in hf_repo:
        return prepare_longform_dataset_old_format
    elif 'modified' in str(subset) and 'synthetic-hallucinations' in hf_repo:
        return prepare_synthetic
    elif 'trivia_qa' in str(subset) or 'triviaqa' in hf_repo:
        return prepare_triviaqa
    else:
        # Default to one-shot pipeline format
        return prepare_longform_dataset


def find_assistant_tokens_slice(
    input_ids: Int[Tensor, "seq_len"], 
    input_str: str, 
    tokenizer: AutoTokenizer
) -> slice:
    """
    Find the slice of tokens that marks the start of the assistant's response.
    
    Args:
        input_ids: Input token IDs
        input_str: Decoded input string
        tokenizer: Tokenizer
        
    Returns:
        slice: Slice indicating assistant response start tokens
    """
    eot_tokens = [
        '<|eot_id|><|start_header_id|>assistant<|end_header_id|>',  # llama 3.1 end-of-turn tokens
        '<|im_start|>assistant',  # qwen end-of-turn tokens
        '<start_of_turn>model',  # gemma end-of-turn tokens
        "[/INST]",  # mistral end-of-turn tokens
    ]
    
    for eot_token in eot_tokens:
        if eot_token in input_str:
            try:
                return find_string_in_tokens(eot_token, input_ids, tokenizer)
            except (AssertionError, ValueError):
                continue
    
    print(f"Could not find assistant tokens in the input_ids: {input_str[:100]}...")
    return slice(0, 0)


def find_string_in_tokens(target: str, tokens: Tensor, tokenizer: AutoTokenizer, max_iters: int = 100) -> slice:
    """
    Performs a binary search to look for a target string inside some tokens.
    
    Args:
        target: String to find
        tokens: Tensor of token IDs
        tokenizer: Tokenizer to decode tokens
        max_iters: Maximum iterations for binary search
        
    Returns:
        slice: Slice indicating where the target's tokens are located
        
    Raises:
        AssertionError: If target is not found in the tokens
        ValueError: If binary search fails to find the target
    """
    assert target in tokenizer.decode(tokens), "The target isn't contained in the whole array of tokens"
    
    # Binary search over the end index of the slice
    n_iters = max_iters
    end_idx_left, end_idx_right = 0, len(tokens) 
    while end_idx_left != end_idx_right and n_iters > 0:
        mid = (end_idx_left + end_idx_right) // 2
        if target in tokenizer.decode(tokens[:mid]):
            end_idx_right = mid
        else:
            end_idx_left = mid + 1
        n_iters -= 1
    end_idx = end_idx_left
    
    # Binary search over the start index of the slice
    n_iters = max_iters
    start_idx_left, start_idx_right = 0, end_idx-1 
    while start_idx_left != start_idx_right and n_iters > 0:
        mid = (start_idx_left + start_idx_right + 1) // 2
        if target in tokenizer.decode(tokens[mid:end_idx]):
            start_idx_left = mid
        else:
            start_idx_right = mid-1
        n_iters -= 1
    start_idx = start_idx_left
    
    target_slice = slice(start_idx, end_idx)
    if target not in tokenizer.decode(tokens[target_slice]):
        raise ValueError(f"Failed to find {target} in tokens: {[tokenizer.decode([tok]) for tok in tokens]}")
    return target_slice


def slice_to_list(slice_obj: slice, length: Optional[int] = None) -> List[int]:
    """
    Convert a slice object to a list of indices.
    
    Args:
        slice_obj: Slice object
        length: Total length (required if stop is None)
        
    Returns:
        List of indices
    """
    start, stop, step = slice_obj.start, slice_obj.stop, slice_obj.step
    
    # If length is not provided, use stop if it's not None, else raise an error
    if length is None:
        if stop is not None:
            length = stop
        else:
            raise ValueError("Length must be provided if stop is None")
    
    # Adjust start, stop, and step
    start = 0 if start is None else start
    stop = length if stop is None else stop
    step = 1 if step is None else step
    
    return list(range(start, stop, step))


In [ ]:
'''
Load datasets.
'''
from types import SimpleNamespace
import datasets

config_defaults = {
    "subset": None,
    "split": "train",
    "max_length": 2048,
    "ignore_buffer": 0,
    "default_ignore": False,
    "last_span_token": False,
    "pos_weight": 1.0,
    "neg_weight": 1.0,
    "shuffle": True,
    "seed": 42,
    "process_on_the_fly": False,
    "max_num_samples": None,
}

train_dataset_config = [
    {
        "dataset_id": "llama3_1_8b_longfact_train",
        "hf_repo": "obalcells/longfact-annotations",
        "subset": "Meta-Llama-3.1-8B-Instruct",
        "split": "train",
        "max_length": 1536,
        "default_ignore": False,
        "last_span_token": False,
        "ignore_buffer": 0,
        "pos_weight": 10.0,
        "neg_weight": 10.0,
        "shuffle": True,
        "seed": 42,
        "process_on_the_fly": False,
    },
    {
        "dataset_id": "llama3_1_8b_longfact_augmented_train",
        "hf_repo": "obalcells/longfact-augmented-annotations",
        "subset": "Meta-Llama-3.1-8B-Instruct",
        "split": "train",
        "max_length": 1536,
        "default_ignore": False,
        "last_span_token": False,
        "ignore_buffer": 0,
        "pos_weight": 10.0,
        "neg_weight": 10.0,
        "shuffle": True,
        "seed": 42,
        "process_on_the_fly": False,
    },
    {
        "dataset_id": "llama3_3_70b_longfact_train",
        "hf_repo": "obalcells/longfact-annotations",
        "subset": "Llama-3.3-70B-Instruct",
        "split": "train",
        "max_length": 1536,
        "default_ignore": False,
        "last_span_token": False,
        "ignore_buffer": 0,
        "pos_weight": 10.0,
        "neg_weight": 10.0,
        "shuffle": True,
        "seed": 42,
        "process_on_the_fly": False,
    },
    {
        "dataset_id": "llama3_3_70b_longfact_augmented_train",
        "hf_repo": "obalcells/longfact-augmented-annotations",
        "subset": "Llama-3.3-70B-Instruct",
        "split": "train",
        "max_length": 1536,
        "default_ignore": False,
        "last_span_token": False,
        "ignore_buffer": 0,
        "pos_weight": 10.0,
        "neg_weight": 10.0,
        "shuffle": True,
        "seed": 42,
        "process_on_the_fly": False,
    },
    {
        "dataset_id": "qwen2_5_7b_longfact_augmented_train",
        "hf_repo": "obalcells/longfact-augmented-annotations",
        "subset": "Qwen2.5-7B-Instruct",
        "split": "train",
        "max_length": 1536,
        "default_ignore": False,
        "last_span_token": False,
        "ignore_buffer": 0,
    },
    {
        "dataset_id": "gemma2_9b_longfact_augmented_train",
        "hf_repo": "obalcells/longfact-augmented-annotations",
        "subset": "gemma-2-9b-it",
        "split": "train",
        "max_length": 1536,
        "default_ignore": False,
        "last_span_token": False,
        "ignore_buffer": 0,
        "pos_weight": 10.0,
        "neg_weight": 10.0,
        "shuffle": True,
        "seed": 42,
        "process_on_the_fly": False,
    },
    {
        "dataset_id": "mistral_small_24b_longfact_augmented_train",
        "hf_repo": "obalcells/longfact-augmented-annotations",
        "subset": "Mistral-Small-24B-Instruct-2501",
        "split": "train",
        "max_length": 1536,
        "default_ignore": False,
        "last_span_token": False,
        "ignore_buffer": 0,
        "pos_weight": 10.0,
        "neg_weight": 10.0,
        "shuffle": True,
        "seed": 42,
        "process_on_the_fly": False,
    },
    {
        "dataset_id": "llama3_1_8b_trivia_qa_train",
        "hf_repo": "obalcells/triviaqa-balanced",
        "subset": "Meta-Llama-3.1-8B-Instruct",
        "split": "train",
        "max_length": 1536,
        "default_ignore": True,
        "last_span_token": False,
        "ignore_buffer": 0,
        "pos_weight": 10.0,
        "neg_weight": 10.0,
        "shuffle": True,
        "seed": 42,
        "process_on_the_fly": False,
    },
]

eval_dataset_config = [
    {
        "dataset_id": "llama3_1_8b_longform_test",
        "hf_repo": "obalcells/longfact-annotations",
        "subset": "Meta-Llama-3.1-8B-Instruct",
        "split": "test",
        "max_length": 1536,
        "pos_weight": 10.0,
        "neg_weight": 10.0,
        "default_ignore": False,
        "shuffle": False,
    },
    {
        "dataset_id": "llama3_1_8b_longform_augmented_test",
        "hf_repo": "obalcells/longfact-augmented-annotations",
        "subset": "Meta-Llama-3.1-8B-Instruct",
        "split": "test",
        "max_length": 1536,
        "pos_weight": 10.0,
        "neg_weight": 10.0,
        "default_ignore": False,
        "shuffle": False,
    },
    {
        "dataset_id": "llama3_1_8b_trivia_qa_test",
        "hf_repo": "obalcells/triviaqa-balanced",
        "subset": "Meta-Llama-3.1-8B-Instruct",
        "split": "test",
        "max_length": 128,
        "pos_weight": 10.0,
        "neg_weight": 10.0,
        "default_ignore": False,
        "shuffle": False,
    },
    {
        "dataset_id": "llama3_1_8b_healthbench_test",
        "hf_repo": "obalcells/healthbench-annotations",
        "subset": "Meta-Llama-3.1-8B-Instruct",
        "split": "test",
        "max_length": 1536,
        "pos_weight": 1.0,
        "neg_weight": 1.0,
        "default_ignore": False,
        "shuffle": False,
    },
]

for i, entry in enumerate(train_dataset_config):
    train_dataset_config[i] = SimpleNamespace(**dict(config_defaults, **entry))
for i, entry in enumerate(eval_dataset_config):
    eval_dataset_config[i] = SimpleNamespace(**dict(config_defaults, **entry))


def create_probing_dataset(cfg, tokenizer) -> TokenizedProbingDataset:
    """
    Create a probing dataset from configuration.
    
    This loads the dataset from HuggingFace and processes it using the
    appropriate dataset-specific preparation function.
    """
    # Load dataset from HuggingFace
    if cfg.subset:
        raw_hf_dataset = datasets.load_dataset(cfg.hf_repo, cfg.subset, split=cfg.split)
    else:
        raw_hf_dataset = datasets.load_dataset(cfg.hf_repo, split=cfg.split)

    print(f"Loading dataset: {cfg.hf_repo} | {cfg.subset} | {cfg.split}")

    # Get appropriate preparation function
    prepare_function = get_prepare_function(cfg.hf_repo, cfg.subset)

    # Convert HF dataset to list of probing items
    probing_items: List[ProbingItem] = prepare_function(raw_hf_dataset)

    tokenized_probing_dataset = TokenizedProbingDataset(
        items=probing_items,
        config=cfg,
        tokenizer=tokenizer)

    return tokenized_probing_dataset

print("Loading datasets:")
train_datasets: List[TokenizedProbingDataset] = [
    create_probing_dataset(config, tokenizer)
    for config in train_dataset_config]
eval_datasets: List[TokenizedProbingDataset] = [
    create_probing_dataset(config, tokenizer)
    for config in eval_dataset_config]

# Concatenate training datasets
train_dataset = train_datasets[0]
for dataset in train_datasets[1:]:
    train_dataset += dataset

In [8]:
from transformers import TrainingArguments

training_config = SimpleNamespace(
    upload_to_hf=False,
    save_evaluation_metrics=True,
    save_roc_curves=False,
    dump_raw_eval_results=False,
    # Training hyperparameters
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    high_loss_threshold=None,
    lambda_lm=0.0,
    lambda_kl=0.0,  # Down from 0.5 since we're not doing LoRA
    anneal_max_aggr=True,
    anneal_warmup=1.0,
    probe_head_lr=1e-3,
    lora_lr=1e-4,
    enable_gradient_checkpointing=True,
    gradient_accumulation_steps=1,
    max_grad_norm=1.0,
    logging_steps=10,
    seed=42,
    # DEFAULTS
    eval_steps=-1,
    sparsity_penalty_weight=None,
    max_steps=-1,
    num_train_epochs=1,
    learning_rate=5e-5,
    probe_config=SimpleNamespace(
        threshold=0.5,
        probe_path=Path("value_head_probes/linear"),
    ),
)


training_args = TrainingArguments(
    output_dir="output",
    per_device_train_batch_size=training_config.per_device_train_batch_size,
    per_device_eval_batch_size=training_config.per_device_eval_batch_size,
    max_steps=training_config.max_steps,
    num_train_epochs=training_config.num_train_epochs,
    logging_steps=10,
    eval_steps=training_config.eval_steps,
    remove_unused_columns=False,
    label_names=["classification_labels", "lm_labels"],
    #report_to="wandb",
    #run_name=training_config.probe_config.probe_id,
    eval_strategy="no",
    logging_first_step=True,
    logging_strategy="steps",
    max_grad_norm=1.0,
    gradient_accumulation_steps=1,
    learning_rate=training_config.learning_rate,
    seed=42,
    disable_tqdm=True,
)

# Add separate learning rates to training_args
training_args.probe_head_lr = 1e-3
training_args.lora_lr = 1e-4

# Disable checkpoint saving
# (there's a weird bug that occurs when trying to save during training)
training_args.set_save(strategy="no");

In [9]:
"""Loss functions for probe training."""

from typing import List, Optional, Tuple, Union

import torch
import torch.nn.functional as F
from jaxtyping import Float, Int
from torch import Tensor
from peft import PeftModel
from transformers import AutoModelForCausalLM


def compute_probe_bce_loss(
    probe_logits: Float[Tensor, 'batch_size seq_len'],
    classification_labels: Float[Tensor, 'batch_size seq_len'],
    classification_weights: Float[Tensor, 'batch_size seq_len'],
    max_clipped_logits: float = 100.0,
    ignore_label: float = -100.0,
):
    # Clip logits to prevent extreme values
    probe_logits_clipped = torch.clamp(
        probe_logits,
        min=-max_clipped_logits,
        max=max_clipped_logits
    )
    try:
        bce_loss = F.binary_cross_entropy_with_logits(
            probe_logits_clipped,
            classification_labels,
            weight=classification_weights,
            reduction='none'
        )
        bce_loss = bce_loss[(classification_labels != ignore_label)].mean()
            
        # Check for NaN in bce_loss
        if torch.isnan(bce_loss):
            print(f"WARNING: NaN detected in bce_loss")
            bce_loss = torch.tensor(0.0, device=probe_logits.device)
        return bce_loss
    except Exception as e:
        print(f"Error in compute_probe_bce_loss: {e}")
        return torch.tensor(0.0, device=probe_logits.device)


def compute_probe_max_aggregation_loss(
    probe_logits: Float[Tensor, 'batch_size seq_len'],
    classification_labels: Float[Tensor, 'batch_size seq_len'],
    classification_weights: Float[Tensor, 'batch_size seq_len'],
    positive_spans: List[List[Tuple[int, int]]],
    negative_spans: List[List[Tuple[int, int]]],
    max_clipped_logits: float = 100.0,
    sparsity_penalty_weight: Optional[float] = None,
):
    """
    Computes the span-level max-aggregation loss.
    For positive spans, loss is BCE(max(logits_in_span), 1.0).
    For negative spans, loss is BCE(max(logits_in_span), 0.0).
    The final loss is the mean over all spans in the batch.
    """

    span_losses = []
    device = probe_logits.device
    dtype = probe_logits.dtype

    # Clip logits to prevent extreme values
    probe_logits_clipped = torch.clamp(
        probe_logits,
        min=-max_clipped_logits,
        max=max_clipped_logits
    )

    for i in range(probe_logits_clipped.shape[0]): # Iterate over batch items
        # Positive spans
        for start, end in positive_spans[i]:
            should_ignore = (classification_labels[i, start:end+1] == -100.0).any()

            if should_ignore:
                continue

            span_logits = probe_logits_clipped[i, start:end+1]

            max_logit = torch.max(span_logits)
            target = torch.tensor(1.0, device=device, dtype=dtype)
            loss = F.binary_cross_entropy_with_logits(max_logit, target, reduction='none')
            span_losses.append(loss)

        # Negative spans
        for start, end in negative_spans[i]:
            should_ignore = (classification_labels[i, start:end+1] == -100.0).any()

            if should_ignore:
                continue

            span_logits = probe_logits_clipped[i, start:end+1]

            max_logit = torch.max(span_logits)
            target = torch.tensor(0.0, device=device, dtype=dtype)
            loss = F.binary_cross_entropy_with_logits(max_logit, target, reduction='none')
            span_losses.append(loss)

    if not span_losses:
        # No valid spans found in the batch, return zero loss
        return torch.tensor(0.0, device=device, dtype=dtype)

    # Compute the mean loss over all spans in the batch
    final_loss = torch.mean(torch.stack(span_losses))

    if sparsity_penalty_weight is not None:
        sparsity_loss = compute_sparsity_loss(
            probe_logits=probe_logits,
            classification_labels=classification_labels,
        )

        final_loss = final_loss + sparsity_penalty_weight * sparsity_loss

    # Check for NaN
    if torch.isnan(final_loss):
        print(f"WARNING: NaN detected in compute_probe_max_aggregation_loss. Returning 0.0")
        final_loss = torch.tensor(0.0, device=device, dtype=dtype)

    return final_loss


def compute_sparsity_loss(
    probe_logits: Float[Tensor, "batch seq_len 1"],
    attention_mask: Int[Tensor, "batch seq_len"],
) -> Float[Tensor, ""]:
    """
    Compute sparsity loss to encourage probe to be selective.
    
    This loss encourages the probe to have low average activation,
    preventing it from flagging everything as a hallucination.
    
    Args:
        probe_logits: Probe output logits
        attention_mask: Mask for valid tokens
    
    Returns:
        Scalar sparsity loss
    """
    # Get probabilities
    probe_probs = torch.sigmoid(probe_logits.squeeze(-1))  # [batch, seq_len]
    
    # Apply attention mask
    masked_probs = probe_probs * attention_mask
    
    # Compute average activation
    num_valid_tokens = attention_mask.sum()
    if num_valid_tokens == 0:
        return torch.tensor(0.0, device=probe_logits.device)
    
    avg_activation = masked_probs.sum() / num_valid_tokens
    
    # Sparsity loss is just the average activation
    return avg_activation


def mask_high_loss_spans(
    lm_model: Union[AutoModelForCausalLM, PeftModel],
    input_ids: Float[Tensor, 'batch_size seq_len'],
    attention_mask: Float[Tensor, 'batch_size seq_len'],
    classification_labels: Float[Tensor, 'batch_size seq_len'],
    spans: List[List[Tuple[int, int]]],
    threshold: float = 1.0, # threshold for the loss to be considered high
):
    """
    Computes the span-level max-aggregation loss.
    For positive spans, loss is BCE(max(logits_in_span), 1.0).
    For negative spans, loss is BCE(max(logits_in_span), 0.0).
    The final loss is the mean over all spans in the batch.
    """

    if isinstance(lm_model, PeftModel):
        with lm_model.disable_adapter():
            lm_logits: Float[Tensor, 'batch_size seq_len vocab_size'] = lm_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=False,
            ).logits
    else:
        lm_logits: Float[Tensor, 'batch_size seq_len vocab_size'] = lm_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=False,
        ).logits

    log_probs = torch.nn.functional.log_softmax(lm_logits, dim=-1)
    log_probs: Float[Tensor, 'batch_size seq_len'] = log_probs.gather(-1, input_ids[:, 1:].unsqueeze(-1)).squeeze(-1)

    for i in range(log_probs.shape[0]): # Iterate over batch items
        # Do it only for the supported entity spans
        for start, end in spans[i]:
            if start > end: # Invalid span, skip
                continue

            span_neg_log_probs = -log_probs[i, start:end+1]
            max_neg_log_prob = span_neg_log_probs.max()

            if max_neg_log_prob > threshold:
                classification_labels[i, start:end+1] = -100.0

    return classification_labels


def compute_kl_divergence_loss(
    model: 'ValueHeadProbe',
    lm_logits: Float[Tensor, "batch seq_len vocab_size"],
    input_ids: Int[Tensor, "batch seq_len"],
    attention_mask: Int[Tensor, "batch seq_len"],
    lm_labels: Int[Tensor, "batch seq_len"],
) -> Float[Tensor, ""]:
    """
    Compute KL divergence between model with LoRA and base model.
    
    Args:
        model: The ValueHeadProbe model
        lm_logits: Logits from model with LoRA adapters
        input_ids: Input token IDs
        attention_mask: Attention mask
        lm_labels: Language modeling labels
        
    Returns:
        Scalar KL divergence loss
    """
    
    # Check if model has LoRA adapters
    if not isinstance(model.model, PeftModel) or not model.model.active_adapters:
        return torch.tensor(0.0, device=lm_logits.device)
    
    # Get base model outputs without LoRA
    with model.model.disable_adapter():
        base_outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        base_logits = base_outputs["lm_logits"].detach()
    
    # Compute KL divergence
    with torch.autocast(device_type=lm_logits.device.type, enabled=False):
        log_q = torch.log_softmax(lm_logits.float(), -1)  # model with LoRA
        p_ref = torch.softmax(base_logits.float(), -1)    # reference model
        kl = F.kl_div(log_q, p_ref, reduction='none', log_target=False).sum(-1)
        
        # Only compute loss on valid tokens
        active_mask = (lm_labels != -100)
        if not active_mask.any():
            return torch.tensor(0.0, device=lm_logits.device)
            
        kl_loss = kl[active_mask].mean()
    
    return kl_loss

In [10]:
from transformers import TrainerCallback
import time

class ProgressPrinter(TrainerCallback):
    def __init__(self, print_every=1):
        self.print_every = print_every
        self.start_time = None
        self.step_times = []
        self.last_step_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        self.last_step_time = self.start_time
        total = state.max_steps if state.max_steps > 0 else "?"
        print(f"[progress] training started, total steps: {total}", flush=True)

    def on_step_end(self, args, state, control, **kwargs):
        now = time.time()
        step_dur = now - self.last_step_time
        self.last_step_time = now
        self.step_times.append(step_dur)
        # keep last 50 for a rolling average
        if len(self.step_times) > 50:
            self.step_times.pop(0)

        if state.global_step % self.print_every != 0:
            return

        avg = sum(self.step_times) / len(self.step_times)
        elapsed = now - self.start_time
        total = state.max_steps
        if total and total > 0:
            remaining = (total - state.global_step) * avg
            pct = 100.0 * state.global_step / total
            print(
                f"[progress] step {state.global_step}/{total} "
                f"({pct:.1f}%) | "
                f"avg {avg:.2f}s/step | "
                f"elapsed {elapsed/60:.1f}min | "
                f"ETA {remaining/60:.1f}min",
                flush=True,
            )
        else:
            print(
                f"[progress] step {state.global_step} | "
                f"avg {avg:.2f}s/step | "
                f"elapsed {elapsed/60:.1f}min",
                flush=True,
            )

    def on_log(self, args, state, control, logs=None, **kwargs):
        # Also surface the loss values that Trainer logs
        if logs and "loss" in logs:
            print(f"[log] step {state.global_step}: {logs}", flush=True)

In [11]:
"""File I/O utilities."""

import json
import yaml
import numpy as np
import torch
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Union
import os


def default_serializer(obj: Any) -> Any:
    """
    Convert non-serializable objects to JSON-serializable types.
    
    Handles common ML types like numpy arrays, torch tensors, dtypes, etc.
    
    Args:
        obj: Object to convert
        
    Returns:
        JSON-serializable version of the object
    """
    if isinstance(obj, torch.dtype):
        return str(obj)
    elif isinstance(obj, (np.ndarray, torch.Tensor)):
        return obj.cpu().tolist()
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, Path):
        return str(obj)
    elif hasattr(obj, "__dataclass_fields__"):
        return {k: default_serializer(v) for k, v in obj.__dict__.items()}
    elif hasattr(obj, '__dict__'):
        return obj.__dict__
    else:
        return obj

def dataclass_to_dict(obj) -> dict:
    """
    Converts a dataclass to a dictionary. Will recurse through
    lists, dicts, and nested dataclasses.
    """

    if hasattr(obj, "__dataclass_fields__"):
        return {k: dataclass_to_dict(v) for k, v in obj.__dict__.items()}
    elif isinstance(obj, list):
        return [dataclass_to_dict(v) for v in obj]
    elif isinstance(obj, dict):
        return {k: dataclass_to_dict(v) for k, v in obj.items()}
    else:
        return obj

def pydantic_to_dict(obj) -> dict:
    """
    Converts a Pydantic model to a dictionary using model_dump().
    """
    if hasattr(obj, 'model_dump'):
        return obj.model_dump()
    elif isinstance(obj, list):
        return [pydantic_to_dict(v) for v in obj]
    elif isinstance(obj, dict):
        return {k: pydantic_to_dict(v) for k, v in obj.items()}
    else:
        return obj

def make_directory_wrapped(filepath: str, **kwargs) -> None:
    if isinstance(filepath, Path):
        parent_dir = filepath.parent
    else:
        parent_dir = "/".join(filepath.split("/")[:-1])
    os.makedirs(parent_dir, exist_ok=True, **kwargs)

def save_jsonl(
    dict_list: List[Dict[Any, Any]],
    filepath: str,
    append: bool = False,
    serialize_dataclasses: bool = False,
    serialize_pydantic: bool = False,
) -> None:
    """Write a list of dictionaries to a jsonlines file."""
    make_directory_wrapped(filepath)

    if isinstance(dict_list, dict):
        dict_list = [dict_list]

    if append and (not os.path.exists(filepath)):
        print(f"File {filepath} does not exist, setting append to False")
        append = False

    mode = "a" if append else "w"
    with open(filepath, mode=mode) as file:
        for item in dict_list:
            if serialize_dataclasses:
                item = dataclass_to_dict(item)
            elif serialize_pydantic:
                item = pydantic_to_dict(item)
            file.write(json.dumps(item) + "\n")


def load_jsonl(path: Union[str, Path]) -> List[Dict[str, Any]]:
    """
    Load data from a JSONL file.
    
    Args:
        path: Path to the JSONL file
        
    Returns:
        List of dictionaries
    """
    path = Path(path)
    data = []
    
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    
    return data


def save_json(
    data: Any, 
    path: Union[str, Path], 
    indent: int = 2,
    serializer: Callable = default_serializer
) -> None:
    """
    Save data to a JSON file.
    
    Args:
        data: Data to save
        path: Path to save the file
        indent: Indentation level for pretty printing
        serializer: Serializer function for special types
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(path, 'w', encoding='utf-8') as f:
        if serializer:
            json.dump(data, f, ensure_ascii=False, indent=indent, default=serializer)
        else:
            json.dump(data, f, ensure_ascii=False, indent=indent)


def load_json(path: Union[str, Path]) -> Any:
    """
    Load data from a JSON file.
    
    Args:
        path: Path to the JSON file
        
    Returns:
        Loaded data
    """
    path = Path(path)
    
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def load_yaml(path: Union[str, Path]) -> Dict[str, Any]:
    """
    Load data from a YAML file.
    
    Args:
        path: Path to the YAML file
        
    Returns:
        Dictionary with loaded data
    """
    path = Path(path)
    
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)


In [ ]:
import atexit
from transformers import Trainer
from torch.optim import AdamW
import gc


class ProbeTrainer(Trainer):
    """
    A custom Trainer that merges standard LM next-token-prediction loss (CE)
    with a classification BCE from a 'probe' that hooks an internal layer.
    """
    def __init__(
        self,
        probe: ValueHeadProbe,
        eval_datasets: List[TokenizedProbingDataset],
        cfg,
        eval_steps: Optional[int] = None,
        tokenizer: AutoTokenizer = None,
        **kwargs
    ):
        super().__init__(model=probe, **kwargs)
        self.lambda_lm: float = cfg.lambda_lm
        self.lambda_kl: float = cfg.lambda_kl
        self.anneal_max_aggr: bool = cfg.anneal_max_aggr
        self.anneal_warmup: float = cfg.anneal_warmup
        self.eval_datasets: List[TokenizedProbingDataset] = eval_datasets
        self.threshold = cfg.probe_config.threshold
        self.ignore_label: float = -100.0
        self.gradient_accumulation_steps: int = cfg.gradient_accumulation_steps
        self.eval_steps: Optional[int] = eval_steps
        self.probe_dir: Path = cfg.probe_config.probe_path
        self.tokenizer: AutoTokenizer = tokenizer
        self.high_loss_threshold: Optional[float] = cfg.high_loss_threshold
        self.sparsity_penalty_weight: float = cfg.sparsity_penalty_weight
        self._last_eval_metrics: Optional[dict] = None

    def get_training_progress(self) -> float:
        """Get the current training progress as a float between 0 and 1."""
        if self.state.max_steps is None or self.state.max_steps == 0:
            return 1.0
        return min(1.0, self.state.global_step / self.state.max_steps)
    
    def compute_loss(
        self,
        model: ValueHeadProbe,
        batch: dict,
        return_outputs=False,
        num_items_in_batch=None
    ):
        

        # Get the device from the underlying model if using DataParallel
        device = model.module.device if isinstance(model, nn.DataParallel) else model.device

        input_ids: torch.Tensor = batch["input_ids"].to(device)
        attention_mask: torch.Tensor = batch["attention_mask"].to(device)
        classification_labels: torch.Tensor = batch["classification_labels"].to(device)
        classification_weights: torch.Tensor = batch["classification_weights"].to(device)
        lm_labels: torch.Tensor = batch["lm_labels"].to(device)
        pos_spans: List[List[Tuple[int, int]]] = batch["pos_spans"]
        neg_spans: List[List[Tuple[int, int]]] = batch["neg_spans"]


        # The underlying HF LM expects "labels" for next-token-prediction
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=lm_labels,  # these are LM labels (not probe labels!)
        )

        lm_logits = outputs["lm_logits"]
        probe_logits = outputs["probe_logits"].squeeze(-1)  # shape [B, T]
        lm_loss = outputs["lm_loss"]  # standard next-token CE loss

        if torch.isnan(lm_loss):
            print(f"WARNING: NaN detected in lm_loss")
            lm_loss = torch.tensor(0.0, device=device)

        # Compute KL divergence if needed
        kl_loss = torch.tensor(0., device=device)
        if self.lambda_kl > 0:
            kl_loss = compute_kl_divergence_loss(
                model=model,
                lm_logits=lm_logits,
                input_ids=input_ids,
                attention_mask=attention_mask,
                lm_labels=lm_labels,
            )

        # Mask high-loss spans if configured
        if self.high_loss_threshold is not None:
            classification_labels = mask_high_loss_spans(
                lm_model=model.model,
                input_ids=input_ids,
                attention_mask=attention_mask,
                classification_labels=classification_labels,
                spans=neg_spans,
                threshold=self.high_loss_threshold,
            )

        # Compute probe loss
        probe_loss = compute_probe_bce_loss(
            probe_logits=probe_logits,
            classification_labels=classification_labels,
            classification_weights=classification_weights,
        )

        # Span-level max aggregation loss (if enabled)
        if self.anneal_max_aggr:
            max_aggr_probe_loss = compute_probe_max_aggregation_loss(
                probe_logits=probe_logits,
                classification_labels=classification_labels,
                classification_weights=classification_weights,
                positive_spans=pos_spans,
                negative_spans=neg_spans,
            )
            
            omega = min(1.0, self.get_training_progress() / self.anneal_warmup)
            probe_loss = (1 - omega) * probe_loss + omega * max_aggr_probe_loss
        else:
            omega = 0.0
            max_aggr_probe_loss = torch.tensor(0.0, device=device)

        # Combine losses
        loss = (
            self.lambda_lm * lm_loss +
            self.lambda_kl * kl_loss +
            (1 - self.lambda_lm - self.lambda_kl) * probe_loss
        )

        log_dict = {
            'loss': float(probe_loss.detach().float().item()),
            'lm_loss': float(lm_loss.detach().float().item()),
            'kl_loss': float(kl_loss.detach().float().item()),
            'lambda_lm': self.lambda_lm,
            'lambda_kl': self.lambda_kl,
            'omega': float(omega),
            'active_positions': int(torch.sum((classification_labels != self.ignore_label)).item()),
        }

        if self.anneal_max_aggr:
            log_dict['max_aggr_probe_loss'] = float(max_aggr_probe_loss.detach().float().item())

        self.log(log_dict)

        if return_outputs:
            outputs["loss"] = loss
            outputs["probe_loss"] = probe_loss
            outputs["lm_loss"] = lm_loss
            return (loss, outputs)
        
        # Clean up if not returning outputs
        del outputs, lm_logits, probe_logits
        gc.collect()
        torch.cuda.empty_cache()

        return loss
    
    def create_optimizer(self):
        """
        Create optimizer with separate learning rates for probe head and LoRA adapters.
        """
        
        # Get the device from the underlying model if using DataParallel
        model = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
        
        # Separate parameters into probe head and LoRA groups
        probe_head_params = []
        lora_params = []
        other_params = []
        probe_head_added = False
        
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
                
            if 'value_head' in name:
                probe_head_params.append(param)
                probe_head_added = True
            elif 'lora' in name.lower():
                lora_params.append(param)
            else:
                other_params.append(param)

        assert probe_head_added == True, f"Probe head not found when computing list of trainable parameters"
        
        # Create parameter groups with different learning rates
        param_groups = []
        
        if probe_head_params:
            param_groups.append({
                'params': probe_head_params,
                'lr': self.args.probe_head_lr,
                'name': 'probe_head'
            })
            
        if lora_params:
            param_groups.append({
                'params': lora_params, 
                'lr': self.args.lora_lr,
                'name': 'lora'
            })
            
        if other_params:
            # Fall back to default learning rate for any other parameters
            param_groups.append({
                'params': other_params,
                'lr': self.args.learning_rate,
                'name': 'other'
            })
        
        # Print parameter group info
        print("\n=== Optimizer Parameter Groups ===")
        for i, group in enumerate(param_groups):
            param_count = sum(p.numel() for p in group['params'])
            print(f"Group {i} ({group['name']}): {param_count:,} parameters, lr={group['lr']}")

        print(f"lora_lr: {self.args.lora_lr} (type={type(self.args.lora_lr)})")
        print(f"probe_head_lr: {self.args.probe_head_lr} (type={type(self.args.probe_head_lr)})")
        
        optimizer = AdamW(
            param_groups,
            eps=self.args.adam_epsilon if hasattr(self.args, 'adam_epsilon') else 1e-8
        )
        self.optimizer = optimizer
        return optimizer
    
    def create_optimizer_and_scheduler(self, num_training_steps: int):
        """
        Override to ensure our custom optimizer is created before the scheduler.
        """
        # Create our custom optimizer first
        self.optimizer = self.create_optimizer()
        
        # Then create the scheduler using the parent method
        self.create_scheduler(
            num_training_steps=num_training_steps,
            optimizer=self.optimizer
        )

    def evaluate(
        self,
        eval_dataset=None,
        ignore_keys=None,
        metric_key_prefix: str = "eval",
        save_roc_curves: bool = False,
        dump_raw_eval_results: bool = False,
        verbose: bool = False,
    ):
        all_eval_metrics = {}

        # Check if this is a final evaluation
        is_final_evaluation = self.get_training_progress() >= 1.0
        
        # If this is a final evaluation and we've already done it, skip
        if is_final_evaluation and self._last_eval_metrics is not None:
            if verbose:
                print("Final evaluation already completed, skipping duplicate evaluation.")
            return self._last_eval_metrics

        # Evaluate on each dataset
        for dataset in self.eval_datasets:
            eval_dataloader = self.get_eval_dataloader(dataset)

            model = self._wrap_model(
                self.model,
                training=False,
                dataloader=eval_dataloader
            ).eval()

            metrics = evaluate_probe(
                model,
                eval_dataloader,
                threshold=self.threshold,
                metric_key_prefix=dataset.config.dataset_id,
                verbose=False,
                save_roc_curves=save_roc_curves,
                save_dir=self.probe_dir if save_roc_curves else None,
                dump_raw_results=dump_raw_eval_results,
            )

            if verbose:
                print_eval_metrics(metrics, metric_key_prefix=dataset.config.dataset_id)
            
            self.log(metrics)
            all_eval_metrics.update(metrics)

            # Save metrics to JSONL file
            metrics['global_step'] = self.state.global_step
            metrics['training_progress'] = self.get_training_progress()
            metrics['dataset_id'] = dataset.config.dataset_id
            save_jsonl([metrics], self.probe_dir / "eval_metrics.jsonl", append=True)

        # Store the metrics for later retrieval
        self._last_eval_metrics = all_eval_metrics
        return all_eval_metrics


def tokenized_probing_collate_fn(batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    """
    Collate function for DataLoader that handles variable-length tokenized sequences.
    
    Args:
        batch: List of tokenized dataset items
    
    Returns:
        Batched dictionary with padded sequences
    """
    # Find max length in batch
    max_len = max(len(item["input_ids"]) for item in batch)
    
    # Initialize batched tensors
    batch_size = len(batch)
    input_ids = torch.full((batch_size, max_len), 0, dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_len), dtype=torch.long)
    classification_labels = torch.full((batch_size, max_len), -100.0, dtype=torch.float32)
    classification_weights = torch.zeros((batch_size, max_len), dtype=torch.float32)
    lm_labels = torch.full((batch_size, max_len), -100, dtype=torch.long)
    
    # Lists for spans
    pos_spans = []
    neg_spans = []
    
    # Fill in the batch
    for i, item in enumerate(batch):
        seq_len = len(item["input_ids"])
        input_ids[i, :seq_len] = item["input_ids"]
        attention_mask[i, :seq_len] = item["attention_mask"]
        classification_labels[i, :seq_len] = item["classification_labels"]
        classification_weights[i, :seq_len] = item["classification_weights"]
        lm_labels[i, :seq_len] = item["lm_labels"]
        pos_spans.append(item["pos_spans"])
        neg_spans.append(item["neg_spans"])
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "classification_labels": classification_labels,
        "classification_weights": classification_weights,
        "lm_labels": lm_labels,
        "pos_spans": pos_spans,
        "neg_spans": neg_spans,
    }


trainer = ProbeTrainer(
    probe=probe,
    eval_datasets=eval_datasets,
    cfg=training_config,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=None, # this is a dummy argument is for the HF base Trainer class
    data_collator=tokenized_probing_collate_fn,
    eval_steps=training_config.eval_steps,
    tokenizer=tokenizer,
    callbacks=[ProgressPrinter(print_every=1)],
)

def save_model_callback():
    """Save probe weigths, tokenizer and training config to disk."""
    probe.save(training_config.probe_config.probe_path)
    tokenizer.save_pretrained(training_config.probe_config.probe_path)
    save_json(
        training_config,
        training_config.probe_config.probe_path / "training_config.json"
    )

# Register save callback for unexpected exits
atexit.register(save_model_callback)

print("Training...")
trainer.train()

print(f"Saving model to {training_config.probe_config.probe_path}")
save_model_callback()

In [17]:
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve


def compute_clf_metrics(
    preds: np.ndarray,
    labels: np.ndarray, 
    probs: np.ndarray
) -> Dict[str, float]:
    """
    Compute classification metrics.
    
    Args:
        preds: Binary predictions (0 or 1)
        labels: Ground truth labels (0 or 1)
        probs: Probability scores
        
    Returns:
        Dictionary of metrics
    """
    assert all((labels == 0.0) | (labels == 1.0)), "labels must be either 0 or 1"
    
    # Basic classification metrics
    accuracy = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    auc_score = roc_auc_score(labels, probs) if len(np.unique(labels)) == 2 else float('nan')
    
    # Find optimal threshold
    optimal_threshold = float('nan')
    threshold_optimized_accuracy = float('nan')
    recall_at_01_fpr = float('nan')
    
    if len(np.unique(labels)) == 2:
        # ROC curve
        fpr, tpr, thresholds = roc_curve(labels, probs)
        
        # Find optimal threshold for accuracy
        unique_probs = np.unique(probs)
        if len(unique_probs) > 100:
            percentiles = np.linspace(0, 100, 100)
            threshold_candidates = np.percentile(unique_probs, percentiles)
        else:
            threshold_candidates = unique_probs
        
        best_accuracy = 0.0
        optimal_threshold = 0.5
        
        for threshold in threshold_candidates:
            y_pred = (probs >= threshold).astype(int)
            acc = accuracy_score(labels, y_pred)
            if acc > best_accuracy:
                best_accuracy = acc
                optimal_threshold = threshold
        
        threshold_optimized_accuracy = best_accuracy
        
        # Calculate recall at 0.1 FPR
        target_fpr = 0.1
        idx = np.where(fpr <= target_fpr)[0]
        if len(idx) > 0:
            recall_at_01_fpr = tpr[idx[-1]]
        else:
            recall_at_01_fpr = 0.0
    
    # Count distributions
    true_positive_count = int(np.sum(labels == 1.0))
    true_negative_count = int(np.sum(labels == 0.0))
    pred_positive_count = int(np.sum(preds == 1.0))
    pred_negative_count = int(np.sum(preds == 0.0))
    total_samples = len(labels)
    
    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "auc": float(auc_score),
        "optimal_threshold": float(optimal_threshold),
        "threshold_optimized_accuracy": float(threshold_optimized_accuracy),
        "recall_at_0.1_fpr": float(recall_at_01_fpr),
        "true_positive_count": true_positive_count,
        "true_negative_count": true_negative_count,
        "pred_positive_count": pred_positive_count,
        "pred_negative_count": pred_negative_count,
        "total_samples": total_samples
    }


def plot_roc_curves(
    all_preds: Dict[str, List[float]],
    all_labels: Dict[str, List[float]], 
    all_probs: Dict[str, List[float]],
    save_dir: str,
    prefix: Optional[str] = None
) -> None:
    """
    Plot ROC curves for different aggregation levels.
    
    Args:
        all_preds: Predictions for each aggregation level
        all_labels: Labels for each aggregation level
        all_probs: Probabilities for each aggregation level
        save_dir: Directory to save plots
        prefix: Optional prefix for filename
    """
    os.makedirs(save_dir, exist_ok=True)
    
    plt.figure(figsize=(18, 6))
    
    fpr_targets = [0.05, 0.1, 0.2, 0.5]
    dot_color = "black"
    dot_size = 40
    
    for i, agg_level in enumerate(['all', 'span', 'span_max']):
        plt.subplot(1, 3, i+1)
        
        if agg_level not in all_labels or len(all_labels[agg_level]) == 0:
            plt.title(f"{agg_level.replace('_', ' ').title()}\nInsufficient data")
            plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.7)
            continue
        
        labels = np.array(all_labels[agg_level])
        probs = np.array(all_probs[agg_level])
        
        if len(np.unique(labels)) < 2:
            plt.title(f"{agg_level.replace('_', ' ').title()}\nInsufficient label diversity")
            plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.7)
            continue
        
        fpr, tpr, _ = roc_curve(labels, probs)
        roc_auc = roc_auc_score(labels, probs)
        
        plt.fill_between(fpr, tpr, color="#f9c97d", alpha=0.5)
        plt.plot(fpr, tpr, lw=2, color="black", label=f'ROC curve (AUC = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], 'w--', lw=2, alpha=0.7)
        
        # Mark TPR at specific FPRs
        for fpr_target in fpr_targets:
            idx = np.argmin(np.abs(fpr - fpr_target))
            plt.scatter(fpr[idx], tpr[idx], s=dot_size, color=dot_color, zorder=5)
            plt.text(fpr[idx], tpr[idx]+0.03, f"{tpr[idx]:.4f}", fontsize=10, ha="center", color=dot_color)
        
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f"{agg_level.replace('_', ' ').title()}")
        plt.legend(loc="lower right")
        plt.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.7)
    
    plt.tight_layout()
    filename = f"{prefix}_roc_curves.png" if prefix else "roc_curves.png"
    plt.savefig(os.path.join(save_dir, filename), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"ROC curves saved to {os.path.join(save_dir, filename)}")


def print_eval_metrics(
    metrics: dict,
    metric_key_prefix: str = "",
    all_labels: Optional[dict] = None,
    include_random_baseline: bool = True,
    seed: int = 42,
) -> None:
    """
    Print evaluation metrics in a nicely formatted way.
    
    Args:
        metrics: Dictionary of metrics
        metric_key_prefix: Optional prefix for metric keys
        all_labels: Optional labels for computing baselines
        include_random_baseline: Whether to include random baseline
        seed: Random seed for baseline
    """
    if metric_key_prefix:
        print(f"\n===== Evaluation Metrics ({metric_key_prefix}) =====")
    else:
        print("\n===== Evaluation Metrics =====")
    
    prefix = metric_key_prefix + "/" if metric_key_prefix else ""
    
    # Print loss metrics if available
    if f'{prefix}lm_loss' in metrics:
        print("\nLoss Metrics:")
        print(f" - LM Loss:     {metrics.get(f'{prefix}lm_loss', 0):.4f}")
        print(f" - Probe Loss:  {metrics.get(f'{prefix}probe_loss', 0):.4f}")
        print(f" - Sparsity:    {metrics.get(f'{prefix}sparsity', 0):.4f}")
    
    # Print classification metrics for different aggregation levels
    for agg_level in ['all', 'span', 'span_max']:
        if f'{prefix}{agg_level}_accuracy' in metrics:
            print(f"\n{agg_level.replace('_', ' ').title()} - Classification Metrics:")
            print(f" - Accuracy:   {metrics[f'{prefix}{agg_level}_accuracy']:.4f}")
            print(f" - Precision:  {metrics[f'{prefix}{agg_level}_precision']:.4f}")
            print(f" - Recall:     {metrics[f'{prefix}{agg_level}_recall']:.4f}")
            print(f" - F1 Score:   {metrics[f'{prefix}{agg_level}_f1']:.4f}")
            
            if f'{prefix}{agg_level}_auc' in metrics:
                print(f" - AUC:        {metrics[f'{prefix}{agg_level}_auc']:.4f}")
            if f'{prefix}{agg_level}_recall_at_0.1_fpr' in metrics:
                print(f" - Recall @ 0.1 FPR: {metrics[f'{prefix}{agg_level}_recall_at_0.1_fpr']:.4f}")
            if f'{prefix}{agg_level}_threshold_optimized_accuracy' in metrics:
                print(f" - Optimized Accuracy: {metrics[f'{prefix}{agg_level}_threshold_optimized_accuracy']:.4f}")
                print(f"   (Optimal Threshold: {metrics[f'{prefix}{agg_level}_optimal_threshold']:.4f})")
            
            # Print baselines if labels provided
            if all_labels and agg_level in all_labels:
                labels = np.array(all_labels[agg_level])
                if len(labels) > 0:
                    # Majority class baseline
                    majority_class = 1 if np.sum(labels) >= len(labels) / 2 else 0
                    majority_baseline = accuracy_score(labels, np.full_like(labels, majority_class))
                    print(f"    (Majority baseline: {majority_baseline:.4f})")
    
    print("\n==============================\n")


@torch.no_grad()
def evaluate_probe(
    probe: ValueHeadProbe,
    eval_dataloader: DataLoader,
    threshold: float = 0.5,
    metric_key_prefix: Optional[str] = None,
    verbose: bool = True,
    save_roc_curves: bool = True,
    save_dir: Optional[Path] = None,
    dump_raw_results: bool = False,
) -> Dict[str, float]:
    """
    Evaluate a probe on a dataset.
    
    Args:
        probe: The probe to evaluate
        eval_dataloader: DataLoader for evaluation data
        threshold: Classification threshold
        metric_key_prefix: Prefix for metric keys
        verbose: Whether to print metrics
        save_roc_curves: Whether to save ROC curve plots
        save_dir: Directory to save results
        dump_raw_results: Whether to save raw predictions
        
    Returns:
        Dictionary of evaluation metrics
    """
    # Force garbage collection before evaluation
    gc.collect()
    torch.cuda.empty_cache()

    # Initialize metric collections for different aggregation levels
    all_probs = {'all': [], 'span': [], 'span_max': []}
    all_preds = {'all': [], 'span': [], 'span_max': []}
    all_labels = {'all': [], 'span': [], 'span_max': []}

    total_lm_loss = 0
    total_probe_loss = 0
    total_sparsity = 0
    num_batches = 0

    for batch in tqdm(eval_dataloader, desc="Evaluating"):
        input_ids: Int[Tensor, "batch_size seq_len"] = batch["input_ids"].to(probe.device)
        attention_mask: Int[Tensor, "batch_size seq_len"] = batch["attention_mask"].to(probe.device)
        classification_labels: Float[Tensor, "batch_size seq_len"] = batch["classification_labels"].to(probe.device)
        classification_weights: Float[Tensor, "batch_size seq_len"] = batch["classification_weights"].to(probe.device)
        lm_labels: Int[Tensor, "batch_size seq_len"] = batch["lm_labels"].to(probe.device)
        pos_spans: List[List[List[int]]] = batch["pos_spans"]
        neg_spans: List[List[List[int]]] = batch["neg_spans"]

        # Forward pass
        outputs = probe(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=lm_labels,
        )

        probe_logits: Float[Tensor, "batch_size seq_len"] = outputs["probe_logits"].squeeze(-1)
        probe_probs: Float[Tensor, "batch_size seq_len"] = torch.sigmoid(probe_logits).float()
        probe_preds: Float[Tensor, "batch_size seq_len"] = (probe_probs > threshold).float()

        probe_loss = compute_probe_bce_loss(
            probe_logits=probe_logits,
            classification_labels=classification_labels,
            classification_weights=classification_weights,
        )

        # 1. All-token metrics (excluding padding and ignored tokens)
        valid_mask = (attention_mask == 1) & (classification_labels != -100.0)
        all_probs['all'].extend(probe_probs[valid_mask].cpu().numpy())
        all_preds['all'].extend(probe_preds[valid_mask].cpu().numpy())
        all_labels['all'].extend(classification_labels[valid_mask].cpu().numpy())

        # 2. Span-level metrics
        annotated_tokens_mask = torch.zeros_like(input_ids, dtype=torch.bool)
        for i in range(len(input_ids)):
            all_spans: List[List[int]] = pos_spans[i] + neg_spans[i]
            for span_range in all_spans:
                start, end = span_range[0], span_range[1]
                assert start <= end, f"Invalid span range: {span_range}"
                annotated_tokens_mask[i, start:end+1] = True

        # Filter out ignored tokens
        annotated_tokens_mask = annotated_tokens_mask & (classification_labels != -100.0)
        
        all_probs['span'].extend(probe_probs[annotated_tokens_mask].cpu().numpy())
        all_preds['span'].extend(probe_preds[annotated_tokens_mask].cpu().numpy())
        all_labels['span'].extend(classification_labels[annotated_tokens_mask].cpu().numpy())

        # 3. Span-level metrics (with max aggregation)
        for i in range(len(input_ids)):
            all_spans: List[List[int]] = pos_spans[i] + neg_spans[i]
            if len(all_spans) == 0:
                continue

            span_labels = [1.0] * len(pos_spans[i]) + [0.0] * len(neg_spans[i])

            for label, (start, end) in zip(span_labels, all_spans):
                max_prob = probe_probs[i, start:end+1].max().cpu().item()
                max_pred = probe_preds[i, start:end+1].max().cpu().item()
                
                all_probs['span_max'].append(max_prob)
                all_preds['span_max'].append(max_pred)
                all_labels['span_max'].append(label)
        
        # Update running metrics
        total_lm_loss += outputs["lm_loss"].item()
        total_probe_loss += probe_loss.item()
        total_sparsity += probe_probs[attention_mask == 1].mean().item()
        num_batches += 1

    # Compute average metrics
    metrics = {
        "lm_loss": total_lm_loss / num_batches,
        "probe_loss": total_probe_loss / num_batches,
        "sparsity": total_sparsity / num_batches,
        "probe_threshold": threshold,
    }

    # Convert lists to numpy arrays
    all_probs = {k: np.array(v) for k, v in all_probs.items()}
    all_preds = {k: np.array(v) for k, v in all_preds.items()}
    all_labels = {k: np.array(v) for k, v in all_labels.items()}

    # Compute classification metrics for each aggregation level
    for agg_level in ['all', 'span', 'span_max']:
        if len(all_labels[agg_level]) == 0:
            continue

        clf_metrics = compute_clf_metrics(
            all_preds[agg_level], 
            all_labels[agg_level], 
            all_probs[agg_level]
        )

        for metric_name, metric_value in clf_metrics.items():
            metrics[f"{agg_level}_{metric_name}"] = metric_value

    # Add prefix if specified
    if metric_key_prefix:
        metrics = {
            f"{metric_key_prefix}/{k}": v
            for k, v in metrics.items()
        }

    # Print metrics if verbose
    if verbose:
        print_eval_metrics(
            metrics,
            metric_key_prefix=metric_key_prefix,
            all_labels=all_labels,
            include_random_baseline=True,
        )

    # Save ROC curves
    if save_roc_curves and save_dir:
        plot_roc_curves(
            all_preds, 
            all_labels, 
            all_probs, 
            save_dir=save_dir, 
            prefix=metric_key_prefix
        )

    # Save raw results if requested
    if dump_raw_results and save_dir:
        results_dir = save_dir / "eval_results"
        if metric_key_prefix:
            results_dir = results_dir / metric_key_prefix
        results_dir.mkdir(parents=True, exist_ok=True)
        
        with open(results_dir / 'metrics.json', 'w') as f:
            json.dump(metrics, f, indent=4)
        
        # Save raw predictions
        for agg_level in ['span_max']:
            if len(all_labels[agg_level]) > 0:
                np.save(results_dir / f'{agg_level}_probs.npy', all_probs[agg_level])
                np.save(results_dir / f'{agg_level}_preds.npy', all_preds[agg_level])
                np.save(results_dir / f'{agg_level}_labels.npy', all_labels[agg_level])

    # Clean up
    gc.collect()
    torch.cuda.empty_cache()

    return metrics

In [18]:
# This currently fails with an out-of-memory error.

# Final evaluation
eval_metrics = trainer.evaluate(
    save_roc_curves=training_config.save_roc_curves,
    dump_raw_eval_results=training_config.dump_raw_eval_results,
    verbose=True,
)

if training_config.save_evaluation_metrics:
    save_json(
        eval_metrics,
        training_config.probe_config.probe_path / "evaluation_results.json"
    )

Evaluating: 100%|██████████| 248/248 [02:39<00:00,  1.55it/s]



===== Evaluation Metrics (llama3_1_8b_longform_test) =====

Loss Metrics:
 - LM Loss:     0.3737
 - Probe Loss:  0.7613
 - Sparsity:    0.1158

All - Classification Metrics:
 - Accuracy:   0.9487
 - Precision:  0.3807
 - Recall:     0.2837
 - F1 Score:   0.3251
 - AUC:        0.8886
 - Recall @ 0.1 FPR: 0.6384
 - Optimized Accuracy: 0.9570
   (Optimal Threshold: 0.8184)

Span - Classification Metrics:
 - Accuracy:   0.6772
 - Precision:  0.8401
 - Recall:     0.2837
 - F1 Score:   0.4242
 - AUC:        0.7952
 - Recall @ 0.1 FPR: 0.4606
 - Optimized Accuracy: 0.7325
   (Optimal Threshold: 0.2539)

Span Max - Classification Metrics:
 - Accuracy:   0.7706
 - Precision:  0.7606
 - Recall:     0.5140
 - F1 Score:   0.6134
 - AUC:        0.8364
 - Recall @ 0.1 FPR: 0.5381
 - Optimized Accuracy: 0.7762
   (Optimal Threshold: 0.4272)


{'llama3_1_8b_longform_test/lm_loss': '0.3737', 'llama3_1_8b_longform_test/probe_loss': '0.7613', 'llama3_1_8b_longform_test/sparsity': '0.1158', 'llama3_1_8b

Evaluating: 100%|██████████| 227/227 [02:26<00:00,  1.55it/s]



===== Evaluation Metrics (llama3_1_8b_longform_augmented_test) =====

Loss Metrics:
 - LM Loss:     0.4169
 - Probe Loss:  0.9265
 - Sparsity:    0.1240

All - Classification Metrics:
 - Accuracy:   0.9315
 - Precision:  0.3519
 - Recall:     0.2738
 - F1 Score:   0.3080
 - AUC:        0.8702
 - Recall @ 0.1 FPR: 0.5647
 - Optimized Accuracy: 0.9448
   (Optimal Threshold: 0.8240)

Span - Classification Metrics:
 - Accuracy:   0.6537
 - Precision:  0.8743
 - Recall:     0.2738
 - F1 Score:   0.4170
 - AUC:        0.7984
 - Recall @ 0.1 FPR: 0.4745
 - Optimized Accuracy: 0.7255
   (Optimal Threshold: 0.2359)

Span Max - Classification Metrics:
 - Accuracy:   0.7697
 - Precision:  0.7934
 - Recall:     0.5205
 - F1 Score:   0.6286
 - AUC:        0.8495
 - Recall @ 0.1 FPR: 0.5629
 - Optimized Accuracy: 0.7809
   (Optimal Threshold: 0.4079)


{'llama3_1_8b_longform_augmented_test/lm_loss': '0.4169', 'llama3_1_8b_longform_augmented_test/probe_loss': '0.9265', 'llama3_1_8b_longform_augmente

Evaluating: 100%|██████████| 502/502 [00:25<00:00, 19.82it/s]



===== Evaluation Metrics (llama3_1_8b_trivia_qa_test) =====

Loss Metrics:
 - LM Loss:     0.4720
 - Probe Loss:  1.1070
 - Sparsity:    0.2014

All - Classification Metrics:
 - Accuracy:   0.9035
 - Precision:  0.1221
 - Recall:     0.1069
 - F1 Score:   0.1140
 - AUC:        0.6964
 - Recall @ 0.1 FPR: 0.2054
 - Optimized Accuracy: 0.9419
   (Optimal Threshold: 1.0000)

Span - Classification Metrics:
 - Accuracy:   0.5008
 - Precision:  0.9716
 - Recall:     0.1069
 - F1 Score:   0.1927
 - AUC:        0.8291
 - Recall @ 0.1 FPR: 0.5503
 - Optimized Accuracy: 0.7582
   (Optimal Threshold: 0.1541)

Span Max - Classification Metrics:
 - Accuracy:   0.6262
 - Precision:  0.9698
 - Recall:     0.2593
 - F1 Score:   0.4092
 - AUC:        0.8874
 - Recall @ 0.1 FPR: 0.6650
 - Optimized Accuracy: 0.8091
   (Optimal Threshold: 0.2336)


{'llama3_1_8b_trivia_qa_test/lm_loss': '0.472', 'llama3_1_8b_trivia_qa_test/probe_loss': '1.107', 'llama3_1_8b_trivia_qa_test/sparsity': '0.2014', 'llama3_1_

Evaluating: 100%|██████████| 339/339 [03:37<00:00,  1.56it/s]



===== Evaluation Metrics (llama3_1_8b_healthbench_test) =====

Loss Metrics:
 - LM Loss:     0.3816
 - Probe Loss:  0.1197
 - Sparsity:    0.0986

All - Classification Metrics:
 - Accuracy:   0.9699
 - Precision:  0.3867
 - Recall:     0.2719
 - F1 Score:   0.3193
 - AUC:        0.9079
 - Recall @ 0.1 FPR: 0.7189
 - Optimized Accuracy: 0.9745
   (Optimal Threshold: 0.8080)

Span - Classification Metrics:
 - Accuracy:   0.7375
 - Precision:  0.8339
 - Recall:     0.2719
 - F1 Score:   0.4101
 - AUC:        0.8202
 - Recall @ 0.1 FPR: 0.4988
 - Optimized Accuracy: 0.7661
   (Optimal Threshold: 0.2861)

Span Max - Classification Metrics:
 - Accuracy:   0.8155
 - Precision:  0.6996
 - Recall:     0.4449
 - F1 Score:   0.5440
 - AUC:        0.8479
 - Recall @ 0.1 FPR: 0.5506
 - Optimized Accuracy: 0.8167
   (Optimal Threshold: 0.4443)


{'llama3_1_8b_healthbench_test/lm_loss': '0.3816', 'llama3_1_8b_healthbench_test/probe_loss': '0.1197', 'llama3_1_8b_healthbench_test/sparsity': '0.09859',

In [20]:
import gc
import torch

# 1. Drop Python references to anything holding GPU tensors
del trainer       # holds the model, optimiser, scheduler, gradient buffers
del train_dataset, eval_datasets  # tokenised datasets can be large on CPU but harmless

# Any local variables in the previous cell that held tensors should also go.
# If you computed `outputs` or `loss` and they're still bound, del those too.

# 2. Garbage-collect to actually run __del__ on those objects
gc.collect()

# 3. Release cached blocks back to the CUDA allocator
torch.cuda.empty_cache()

# 4. (Optional) Verify
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GiB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GiB")

Allocated: 6.44 GiB
Reserved:  6.53 GiB


In [21]:
import torch
import torch.nn.functional as F
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

PROBE_PATH = training_config.probe_config.probe_path

device = "cuda:0" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
).to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(PROBE_PATH)  # you saved it here in your callback
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Reconstruct the probe from the saved weights
probe = ValueHeadProbe(model, path=PROBE_PATH)
probe.eval()

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

ValueHeadProbe(
  (model): LlamaForCausalLM(
    (model): LlamaModel(
      (embed_tokens): Embedding(128256, 3072)
      (layers): ModuleList(
        (0-27): 28 x LlamaDecoderLayer(
          (self_attn): LlamaAttention(
            (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
            (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
            (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
            (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          )
          (mlp): LlamaMLP(
            (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
            (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
            (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
            (act_fn): SiLUActivation()
          )
          (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
          (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
 

In [22]:
from IPython.display import HTML


@torch.no_grad()
def generate_with_hallucination_scores(
    prompt: str,
    probe: ValueHeadProbe,
    tokenizer,
    max_new_tokens: int = 256,
    temperature: float = 0.7,
    do_sample: bool = True,
):
    model = probe.model
    device = next(model.parameters()).device

    # 1. Build the chat-templated prompt
    messages = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # appends the assistant header
    )
    prompt_ids = tokenizer(prompt_text, return_tensors="pt").input_ids.to(device)
    prompt_len = prompt_ids.shape[1]

    # 2. Generate
    gen_output = model.generate(
        prompt_ids,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        pad_token_id=tokenizer.pad_token_id,
    )
    # gen_output shape: [1, prompt_len + n_generated]

    # 3. One forward pass with the probe hook active to get per-token scores
    full_ids = gen_output  # already includes prompt + generation
    attention_mask = torch.ones_like(full_ids)

    outputs = probe(input_ids=full_ids, attention_mask=attention_mask)
    probe_logits = outputs["probe_logits"].squeeze(-1).squeeze(0)  # [seq_len]
    probs = torch.sigmoid(probe_logits.float())  # [seq_len]

    # 4. Slice to the generated portion only
    gen_ids = full_ids[0, prompt_len:]
    gen_probs = probs[prompt_len:]

    # 5. Decode token-by-token alongside scores
    tokens_with_scores = []
    for tok_id, p in zip(gen_ids.tolist(), gen_probs.tolist()):
        tok_str = tokenizer.decode([tok_id])
        tokens_with_scores.append((tok_str, p))

    full_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return full_text, tokens_with_scores


def chunk_by_sentence(scored_tokens):
    # crude: split on tokens containing '.', '!', '?', '\n'
    chunks, current = [], []
    for tok, p in scored_tokens:
        current.append((tok, p))
        if any(c in tok for c in ".!?\n"):
            chunks.append(current)
            current = []
    if current:
        chunks.append(current)
    return [
        ("".join(t for t, _ in chunk), max(p for _, p in chunk))
        for chunk in chunks
    ]


def render_html(scored_tokens, threshold=0.5):
    from html import escape
    parts = []
    for tok, p in scored_tokens:
        # red intensity scales with probability
        red = int(255 * p)
        bg = f"rgba({red}, 0, 0, {min(p, 0.8):.2f})"
        parts.append(
            f'<span style="background:{bg}" title="{p:.3f}">{escape(tok)}</span>'
        )
    return "".join(parts)

In [23]:
text, scored_tokens = generate_with_hallucination_scores(
    "Who won the 20219 Nobel Prize in Mathematics, and what was it for?",
    probe,
    tokenizer,
)
HTML(render_html(scored_tokens))

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [29]:
text, scored_tokens = generate_with_hallucination_scores(
    "What is the movie 'Her'? Provide as many specific details and examples as possible (such as names of people, numbers, events, locations, dates, times, etc.)",
    probe,
    tokenizer,
)
HTML(render_html(scored_tokens))

In [27]:
text, scored_tokens = generate_with_hallucination_scores(
    "I have a newly diagnosed stage III colon cancer patient. I'm exploring the idea of insulin potentiation therapy with low-dose chemo. Provide a summary of any published data, in order of importance, that confirms or refutes synergy. I'm an oncologist looking for robust evidence.",
    probe,
    tokenizer,
)
HTML(render_html(scored_tokens))